# Customer Discovery Agent: reviewer agent test

Runs the reviewer agent on 70 labelled leads with a local Ollama model and scores its decisions (alert / queue / drop).

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then Runtime > Run all. When cell 2 asks, upload `discovery_project.zip`.
Expect about 5 min of setup and roughly 10 to 20 min for each model run.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Upload the project zip and unpack it
import os, zipfile
from google.colab import files

up = files.upload()
name = next(iter(up))
os.makedirs("/content/project", exist_ok=True)
zipfile.ZipFile(name).extractall("/content/project")
os.chdir("/content/project")
!ls

In [ ]:
# 3. Install Ollama and start the server (about 1-2 min)
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

import subprocess, time, urllib.request
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434", timeout=2)
        print("ollama is up")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama server did not start")

In [ ]:
# 4. Pull the models. qwen2.5:7b is the main one; llama3.1:8b is an optional comparison (cell 9; it scored poorly last time).
!ollama pull qwen2.5:7b
!ollama pull llama3.1:8b
!ollama list

In [ ]:
# 5. Unit tests, then generate the labelled review data
!python -m unittest discover -s tests 2>&1 | tail -4
!python -m discovery.scenarios --per-type 10 --out data/review
# a second, held-out set (different seed) that the prompt was never tuned on
!python -m discovery.scenarios --per-type 10 --seed 99 --out data/heldout

In [ ]:
# 6. Reference points that need no model: the stand-in policy (should be 1.00) and the text-only baseline
!python -m discovery.evaluate --backend policy | head -8

In [ ]:
# 7. Probe: one raw call, to confirm the tool-call format works with this model before the full run
!ollama list
import sys
sys.path.insert(0, "/content/project")
from discovery.ollama_llm import OllamaChat
from discovery.tools import TOOL_SPECS
from discovery.agent import SUBMIT_SPEC, SYSTEM

lead = ('Lead message id 1 from buyer_001 at 2026-10-01T10:00:00:\n'
        '"Need a quote for 50 desks for our office, must order today"\n\nReview this lead.')
reply = OllamaChat(model="qwen2.5:7b").chat(SYSTEM, [{"role": "user", "content": lead}], TOOL_SPECS + [SUBMIT_SPEC])
print("text      :", reply.text)
print("tool calls:", reply.tool_calls)
print("tokens    :", reply.tokens_in, "in /", reply.tokens_out, "out")

In [ ]:
# 8. Full evaluation with qwen2.5:7b (one line per lead; add --limit 15 for a quick trial)
!python -m discovery.evaluate --backend ollama --model qwen2.5:7b -v --save results_qwen.jsonl 2>&1 | tee report_qwen.txt

In [ ]:
# 8b. Held-out check: same model, fresh leads the prompt was never tuned on. This is the honest number.
!python -m discovery.evaluate --backend ollama --model qwen2.5:7b --chat data/heldout_chat.jsonl --labels data/heldout_labels.jsonl -v --save results_qwen_heldout.jsonl 2>&1 | tee report_qwen_heldout.txt

In [ ]:
# 9. Optional comparison with llama3.1:8b on the same leads
!python -m discovery.evaluate --backend ollama --model llama3.1:8b -v --save results_llama.jsonl 2>&1 | tee report_llama.txt

In [ ]:
# 10. Download the reports and per-lead results (or paste the outputs of cells 7, 8 and 9 into the chat)
from google.colab import files
for f in ["report_qwen.txt", "results_qwen.jsonl", "report_qwen_heldout.txt", "results_qwen_heldout.jsonl", "report_llama.txt", "results_llama.jsonl"]:
    if os.path.exists(f):
        files.download(f)